# Aggregated data checks

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = Path(os.environ.get("BREXPERT_DATA_ROOT", REPO_ROOT.parent / "data")).resolve()
print(f"DATA_ROOT = {DATA_ROOT}")

## Load all per-dataset processed CSVs

In [ ]:
import glob

REQUIRED_COLUMNS = ["id", "patient", "dataset", "modality", "birads", "race", "machine",
                    "exam", "segmentation", "context", "findings"]

processed_csvs = sorted(glob.glob(str(DATA_ROOT / "processed" / "**" / "*.csv"), recursive=True))
processed_df = pd.concat([pd.read_csv(p) for p in processed_csvs], ignore_index=True)
print(f"{len(processed_csvs)} processed csvs, {len(processed_df)} rows total")
processed_df["dataset"].value_counts()

## Schema check

In [ ]:
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(processed_df.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"

schema_issues = (
    processed_df.groupby("dataset")
    .agg(
        duplicate_id_rows=("id", lambda s: int(s.duplicated().sum())),
        missing_patient_rows=("patient", lambda s: int(s.isna().sum())),
        missing_exam_path_rows=("exam", lambda s: int(s.isna().sum())),
    )
    .reset_index()
)
schema_issues

## BI-RADS and modality distribution per source

In [ ]:
processed_df.groupby(["dataset", "modality", "birads"], dropna=False).size().unstack(fill_value=0)

## Load the report-generation split CSVs

In [ ]:
from scripts.split import RG, TRAIN_SPLIT_CSV_EXT, VAL_SPLIT_CSV_EXT, TEST_SPLIT_CSV_EXT, convert_birads

split_dir = DATA_ROOT / "report_generation_split"
split_files = {
    "train": split_dir / f"all-{RG}{TRAIN_SPLIT_CSV_EXT}",
    "val": split_dir / f"all-{RG}{VAL_SPLIT_CSV_EXT}",
    "test": split_dir / f"all-{RG}{TEST_SPLIT_CSV_EXT}",
}
split_frames = []
for split_name, path in split_files.items():
    frame = pd.read_csv(path)
    frame["split"] = split_name
    split_frames.append(frame)
splits_df = pd.concat(split_frames, ignore_index=True)
print(f"{len(splits_df)} split rows")
splits_df.head()

## BI-RADS collapse check

In [ ]:
recomputed = splits_df["original_birads"].apply(convert_birads)
mismatches = (recomputed != splits_df["birads"]).sum()
print(f"birads / recomputed mismatches: {mismatches}")
splits_df.groupby(["original_birads", "birads"]).size()

## Split leakage check

In [ ]:
leakage_rows = []
for key in ["patient", "id", "exam"]:
    counts = splits_df.groupby(key)["split"].nunique()
    leakage_rows.append({"identifier": key, "values_in_multiple_splits": int((counts > 1).sum())})
pd.DataFrame(leakage_rows)

## Split completeness check

In [ ]:
processed_ids = set(processed_df["id"].astype(str))
split_ids = set(splits_df["id"].astype(str))
pd.DataFrame([
    ("processed_unique_ids", len(processed_ids)),
    ("split_unique_ids", len(split_ids)),
    ("processed_ids_missing_from_splits", len(processed_ids - split_ids)),
    ("split_ids_missing_from_processed", len(split_ids - processed_ids)),
], columns=["check", "count"])

## Per-modality split files partition the combined split

In [ ]:
modality_totals = []
for split_name, ext in [("train", TRAIN_SPLIT_CSV_EXT), ("val", VAL_SPLIT_CSV_EXT), ("test", TEST_SPLIT_CSV_EXT)]:
    all_rows = len(pd.read_csv(split_dir / f"all-{RG}{ext}"))
    per_modality = 0
    for modality in ["mg", "mr", "us"]:
        path = split_dir / f"{modality}-{RG}{ext}"
        if path.exists():
            per_modality += len(pd.read_csv(path, low_memory=False))
    modality_totals.append({"split": split_name, "all_rg_rows": all_rows, "sum_per_modality_rows": per_modality})
pd.DataFrame(modality_totals)

## Summary

In [ ]:
print(f"Processed rows: {len(processed_df)} across {processed_df['dataset'].nunique()} datasets")
print(f"Split rows: {len(splits_df)} (train={len(splits_df[splits_df.split=='train'])}, "
      f"val={len(splits_df[splits_df.split=='val'])}, test={len(splits_df[splits_df.split=='test'])})")
print(f"Schema issues found: {int(schema_issues[['duplicate_id_rows', 'missing_patient_rows', 'missing_exam_path_rows']].sum().sum())}")
print(f"BI-RADS collapse mismatches: {mismatches}")